# Build `comparisons_df.pickle` (participant-aware baseline dedup + eyetracker replacement)

This notebook constructs a **training-compatible** comparisons dataset used by the pairwise training pipeline.

## Workflow
1. Load **Berlin baseline** from `comparisons_berlin(2).p` (includes `user` and `datetime`).
2. Load **all other datasets** from `scores.csv` (also includes `user` and `datetime`).
3. Remove **participant-level duplicates** in baseline data:
   - Duplicate key: `(dataset, user, unordered_pair(image_l, image_r))`.
   - `(A,B)` is treated as the same comparison as `(B,A)`.
   - Keep the earliest response by `datetime` when available.
   - Report drops **per dataset**.
4. Load `eyetracker_comparisons_df.pickle` (assumed clean) and **replace** matching baseline pairs (swap-safe).
5. Export `comparisons_df.pickle` with a **clean schema** compatible with `train.py/data.py`.
6. Print a final **statistics** cell to validate dataset construction.

## Output schema
The exported pickle contains only:
- `dataset`, `image_l`, `image_r`, `score`, `has_eyetracker`
- plus eyetracker metadata if present: `survey_id`, `trial_id`, `npy_file_l`, `npy_file_r`.

Participant/time fields are used only for cleaning and are not exported.


## 0) Imports and configuration
Edit paths if needed. Run the notebook from the directory that contains your input files.

In [1]:
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd

# ----------------------------
# Input files (edit if needed)
# ----------------------------
SCORES_CSV = Path("scores.csv")
BERLIN_PKL = Path("comparisons_berlin(2).p")
EYE_PKL    = Path("eyetracker_comparisons_df.pickle")

# ----------------------------
# Output file
# ----------------------------
OUT_PKL = Path("comparisons_df.pickle")

# If dataset cannot be inferred from scores.csv paths/URLs, set a fallback or leave None.
KNOWN_DATASET_FALLBACK = None

# Replacement policy: if True, eyetracker comparisons replace baseline comparisons (recommended).
EYE_REPLACES_BASELINE = True


## 1) Helper functions
These helpers normalize URLs/paths, enforce `.jpg` filenames, parse dataset names, and define a swap-safe pair key.

In [2]:
from pathlib import Path

def normalize_path(p):
    # Normalize slashes and strip URL scheme if present.
    if not isinstance(p, str):
        return ""
    s = str(p).strip().replace("\\", "/")
    if s.startswith("http://") or s.startswith("https://"):
        s = urlparse(s).path
    return s

def parse_dataset_and_filename(p):
    # Parse (dataset, filename) from a path or URL. If parsing fails, return (None, None).
    s = normalize_path(p)
    if not s:
        return None, None
    parts = [x for x in s.split("/") if x]
    if len(parts) < 2:
        return None, None
    filename = parts[-1]
    dataset = parts[-2].lower()
    if "." not in filename:
        filename = f"{filename}.jpg"
    return dataset, filename

def ensure_jpg(name):
    # Ensure the value is a .jpg filename; if a path is provided, keep only the basename.
    if not isinstance(name, str):
        return ""
    s = name.strip()
    if not s:
        return ""
    s = s.replace("\\", "/").split("/")[-1]
    if not s.lower().endswith(".jpg"):
        s = f"{Path(s).stem}.jpg"
    return s

def coerce_score(x):
    # Coerce score to {-1, 0, +1}; return None if invalid.
    try:
        v = int(x)
    except Exception:
        return None
    return v if v in (-1, 0, 1) else None

def coerce_datetime(x):
    # Safe datetime parsing; returns Timestamp or NaT.
    return pd.to_datetime(x, errors="coerce", utc=False)

def unordered_pair_key(dataset, image_l, image_r):
    # Swap-safe key: (dataset, min(imgA,imgB), max(imgA,imgB))
    a = ensure_jpg(image_l)
    b = ensure_jpg(image_r)
    if a <= b:
        return (str(dataset).lower(), a, b)
    return (str(dataset).lower(), b, a)


## 2) Load Berlin baseline (`comparisons_berlin(2).p`)
This legacy pickle contains per-response metadata (e.g., `user`, `datetime`). We normalize to a baseline schema and set `has_eyetracker=False`.

In [3]:
ber_raw = pd.read_pickle(BERLIN_PKL)
if not isinstance(ber_raw, pd.DataFrame):
    raise TypeError(f"Expected a pandas DataFrame in {BERLIN_PKL}, got {type(ber_raw)}")

# Identify image columns (legacy pickles commonly use image_i/image_j)
img_cols = None
for a, b in [("image_l", "image_r"), ("image_i", "image_j")]:
    if a in ber_raw.columns and b in ber_raw.columns:
        img_cols = (a, b)
        break
if img_cols is None:
    raise KeyError(f"Could not find image columns in Berlin pickle. Columns: {list(ber_raw.columns)}")

if "score" not in ber_raw.columns:
    raise KeyError("Berlin pickle must contain a 'score' column.")

ber = ber_raw.copy()

# Enforce dataset
if "dataset" not in ber.columns:
    ber["dataset"] = "berlin"
ber["dataset"] = ber["dataset"].astype(str).str.lower()

# Participant/time metadata
if "user" not in ber.columns:
    ber["user"] = "unknown"
if "datetime" not in ber.columns:
    ber["datetime"] = pd.NaT

ber["user"] = ber["user"].astype(str)
ber["datetime"] = ber["datetime"].map(coerce_datetime)

# Normalize core fields
ber["image_l"] = ber[img_cols[0]].map(ensure_jpg)
ber["image_r"] = ber[img_cols[1]].map(ensure_jpg)
ber["score"]   = ber["score"].map(coerce_score)
ber["has_eyetracker"] = False

ber_baseline = ber[["dataset","image_l","image_r","score","user","datetime","has_eyetracker"]].dropna(subset=["score"]).reset_index(drop=True)

print("Berlin baseline loaded:", ber_baseline.shape)
display(ber_baseline.head(10))


Berlin baseline loaded: (7281, 7)


,dataset,image_l,image_r,score,user,datetime,has_eyetracker
0,berlin,209.jpg,7819.jpg,1,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:13:23,False
1,berlin,210.jpg,2123.jpg,1,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:13:33,False
2,berlin,211.jpg,5265.jpg,-1,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:13:43,False
3,berlin,212.jpg,2024.jpg,-1,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:13:52,False
4,berlin,213.jpg,9692.jpg,1,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:14:10,False
5,berlin,214.jpg,5060.jpg,-1,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:14:16,False
6,berlin,216.jpg,1289.jpg,-1,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:14:19,False
7,berlin,217.jpg,5195.jpg,0,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:14:29,False
8,berlin,116.jpg,218.jpg,0,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:14:44,False
9,berlin,219.jpg,8116.jpg,0,cycling9334a308469b956854470ed3668c578f7c99fa3...,2022-09-06 17:15:04,False


## 3) Load `scores.csv` and build baseline for all non-Berlin datasets
This step parses dataset + filenames from the two image paths/URLs in `scores.csv`, normalizes the schema, and excludes Berlin.

In [4]:
scores = pd.read_csv(SCORES_CSV, header=None)
scores = scores.rename(columns={
    0: "datetime",
    1: "user",
    2: "image_l",
    3: "image_r",
    4: "score",
})


def find_first_col(df, options):
    for c in options:
        if c in df.columns:
            return c
    return None

# Column inference lists (extend if your CSV differs)
col_time  = find_first_col(scores, ["datetime", "date", "time", "timestamp"])
col_user  = find_first_col(scores, ["user", "subject", "participant", "session", "userid"])
col_left  = find_first_col(scores, ["image_l", "image_i", "left", "img_l", "url_l", "url_left", "image_left", "image1"])
col_right = find_first_col(scores, ["image_r", "image_j", "right", "img_r", "url_r", "url_right", "image_right", "image2"])
col_score = find_first_col(scores, ["score", "label", "choice", "response"])

if col_left is None or col_right is None or col_score is None:
    raise KeyError(
        "Could not infer required columns from scores.csv.\n"
        f"Columns={list(scores.columns)}\n"
        f"Inferred left={col_left}, right={col_right}, score={col_score}"
    )

sc = scores.copy()
sc["user"] = sc[col_user].astype(str) if col_user else "unknown"
sc["datetime"] = coerce_datetime(sc[col_time]) if col_time else pd.NaT

# Parse dataset + filename
ds_l, fn_l = zip(*sc[col_left].map(parse_dataset_and_filename))
ds_r, fn_r = zip(*sc[col_right].map(parse_dataset_and_filename))

sc["dataset_l"] = list(ds_l)
sc["dataset_r"] = list(ds_r)
sc["image_l"] = [ensure_jpg(x) for x in fn_l]
sc["image_r"] = [ensure_jpg(x) for x in fn_r]

sc["dataset"] = pd.Series(sc["dataset_l"]).fillna(pd.Series(sc["dataset_r"]))
if KNOWN_DATASET_FALLBACK is not None:
    sc["dataset"] = sc["dataset"].fillna(str(KNOWN_DATASET_FALLBACK).lower())
sc["dataset"] = sc["dataset"].astype(str).str.lower()

sc["score"] = sc[col_score].map(coerce_score)
sc["has_eyetracker"] = False

baseline_non_berlin = sc[["dataset","image_l","image_r","score","user","datetime","has_eyetracker"]].dropna(subset=["score"]).copy()
baseline_non_berlin = baseline_non_berlin[baseline_non_berlin["dataset"] != "berlin"].reset_index(drop=True)

print("Non-Berlin baseline loaded from scores.csv:", baseline_non_berlin.shape)
print("\nDatasets (non-Berlin) found in scores.csv:")
display(baseline_non_berlin["dataset"].value_counts().to_frame("n_rows"))
display(baseline_non_berlin.head(10))


Non-Berlin baseline loaded from scores.csv: (6328, 7)

Datasets (non-Berlin) found in scores.csv:


,n_rows
dataset,
sequences,2961
barcelona,1160
london_uk_gov,562
london_uk_collideoscope,561
paris,549
munich,535


,dataset,image_l,image_r,score,user,datetime,has_eyetracker
0,sequences,fi_0_1.jpg,fi_0_2.jpg,1,migueltest,2022-09-26 08:40:54,False
1,sequences,li_4_5.jpg,li_4_6.jpg,0,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:35:46,False
2,london_uk_gov,9.jpg,10.jpg,0,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:35:59,False
3,barcelona,1.jpg,795.jpg,-1,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:36:08,False
4,sequences,li_0_1.jpg,li_0_2.jpg,0,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:36:56,False
5,sequences,si_0_0.jpg,si_0_1.jpg,0,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:38:16,False
6,london_uk_collideoscope,0.jpg,3.jpg,1,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:39:49,False
7,london_uk_gov,57454.jpg,57750.jpg,0,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:40:36,False
8,sequences,si_4_5.jpg,si_4_6.jpg,0,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:40:41,False
9,london_uk_collideoscope,23836.jpg,24629.jpg,-1,cyclingee65858ab02db80be21b910b76652c086fae44...,2022-09-26 13:41:07,False


## 4) Participant-level duplicate removal (baseline only)
### Duplicate definition
A baseline row is a duplicate if the same participant (`user`) answered the same comparison more than once, using a swap-safe pair.

Key: `(dataset, user, unordered_pair(image_l, image_r))`

### Keep rule
Keep the earliest response by `datetime` where possible.

This cell prints the number of dropped rows per dataset for each baseline source.


In [5]:
def drop_participant_duplicates(df, label):
    x = df.copy()
    x["pair_key"] = x.apply(lambda r: unordered_pair_key(r["dataset"], r["image_l"], r["image_r"]), axis=1)
    x["user"] = x["user"].astype(str)

    # Sort so the earliest datetime is kept first (stable sort preserves order)
    x["_dt_sort"] = x["datetime"].fillna(pd.Timestamp.max)
    x = x.sort_values(["dataset","user","pair_key","_dt_sort"], kind="mergesort").reset_index(drop=True)

    before = len(x)
    keep = ~x.duplicated(subset=["dataset","user","pair_key"], keep="first")
    dropped = before - int(keep.sum())

    print(f"[{label}] Dropped participant-level duplicates: {dropped} / {before}")
    if dropped > 0:
        drops_by_dataset = x.loc[~keep].groupby("dataset").size().sort_values(ascending=False)
        display(drops_by_dataset.to_frame("n_dropped"))
    else:
        print("No participant-level duplicates detected.")

    return x.loc[keep].drop(columns=["pair_key","_dt_sort"]).reset_index(drop=True)

ber_clean = drop_participant_duplicates(ber_baseline, "Berlin baseline")
nonber_clean = drop_participant_duplicates(baseline_non_berlin, "Non-Berlin baseline")

print("Berlin after dedup:", ber_clean.shape)
print("Non-Berlin after dedup:", nonber_clean.shape)


[Berlin baseline] Dropped participant-level duplicates: 11 / 7281


,n_dropped
dataset,
berlin,11


[Non-Berlin baseline] Dropped participant-level duplicates: 51 / 6328


,n_dropped
dataset,
sequences,37
barcelona,7
london_uk_gov,3
london_uk_collideoscope,2
munich,2


Berlin after dedup: (7270, 7)
Non-Berlin after dedup: (6277, 7)


## 5) Combine baseline datasets (Berlin + all other cities)
After participant-level deduplication, we concatenate Berlin baseline with the non-Berlin baseline derived from `scores.csv`.

Participant/time metadata are kept temporarily for auditing and are removed before export.


In [6]:
baseline_all = pd.concat([nonber_clean, ber_clean], ignore_index=True)
print("Baseline (all datasets) after participant-dedup:", baseline_all.shape)
display(baseline_all["dataset"].value_counts().to_frame("n_rows"))


Baseline (all datasets) after participant-dedup: (13547, 7)


,n_rows
dataset,
berlin,7270
sequences,2924
barcelona,1153
london_uk_gov,559
london_uk_collideoscope,559
paris,549
munich,533


## 6) Load eyetracker comparisons
Eyetracker data are assumed clean (no `user/datetime`). We normalize filenames and enforce `has_eyetracker=True`.

Eyetracker rows may include Berlin and other datasets.


In [7]:
eye_raw = pd.read_pickle(EYE_PKL)
if not isinstance(eye_raw, pd.DataFrame):
    raise TypeError(f"Expected a pandas DataFrame in {EYE_PKL}, got {type(eye_raw)}")

required = ["dataset","image_l","image_r","score"]
missing = [c for c in required if c not in eye_raw.columns]
if missing:
    raise KeyError(f"Eyetracker pickle missing required columns: {missing}")

eye = eye_raw.copy()
eye["dataset"] = eye["dataset"].astype(str).str.lower()
eye["image_l"] = eye["image_l"].map(ensure_jpg)
eye["image_r"] = eye["image_r"].map(ensure_jpg)
eye["score"] = eye["score"].map(coerce_score)
eye["has_eyetracker"] = True
eye = eye.dropna(subset=["score"]).reset_index(drop=True)

print("Eyetracker loaded:", eye.shape)
print("\nEyetracker rows by dataset:")
display(eye["dataset"].value_counts().to_frame("n_rows"))
display(eye.head(10))


Eyetracker loaded: (1495, 9)

Eyetracker rows by dataset:


,n_rows
dataset,
berlin,999
sequences,496


,dataset,image_l,image_r,score,has_eyetracker,survey_id,trial_id,npy_file_l,npy_file_r
0,berlin,862.jpg,7255.jpg,-1,True,1,1,survey1_trial1_862_left_eyetrack.npy,survey1_trial1_7255_right_eyetrack.npy
1,sequences,fi_0_5.jpg,fi_0_6.jpg,1,True,1,2,survey1_trial2_fi_0_5_left_eyetrack.npy,survey1_trial2_fi_0_6_right_eyetrack.npy
2,berlin,863.jpg,7231.jpg,-1,True,1,3,survey1_trial3_863_left_eyetrack.npy,survey1_trial3_7231_right_eyetrack.npy
3,berlin,865.jpg,9575.jpg,-1,True,1,4,survey1_trial4_865_left_eyetrack.npy,survey1_trial4_9575_right_eyetrack.npy
4,sequences,fi_1_0.jpg,fi_1_1.jpg,-1,True,1,5,survey1_trial5_fi_1_0_left_eyetrack.npy,survey1_trial5_fi_1_1_right_eyetrack.npy
5,sequences,si_3_5.jpg,si_3_6.jpg,-1,True,1,6,survey1_trial6_si_3_5_left_eyetrack.npy,survey1_trial6_si_3_6_right_eyetrack.npy
6,berlin,867.jpg,4916.jpg,1,True,1,7,survey1_trial7_867_left_eyetrack.npy,survey1_trial7_4916_right_eyetrack.npy
7,sequences,fi_1_1.jpg,fi_1_2.jpg,-1,True,1,8,survey1_trial8_fi_1_1_left_eyetrack.npy,survey1_trial8_fi_1_2_right_eyetrack.npy
8,berlin,868.jpg,1810.jpg,1,True,1,9,survey1_trial9_868_left_eyetrack.npy,survey1_trial9_1810_right_eyetrack.npy
9,berlin,869.jpg,5301.jpg,1,True,1,10,survey1_trial10_869_left_eyetrack.npy,survey1_trial10_5301_right_eyetrack.npy


## 7) Replace baseline rows with eyetracker rows (swap-safe)
If `EYE_REPLACES_BASELINE=True`, any baseline row whose swap-safe pair appears in eyetracker is removed.
The eyetracker rows are then appended.

This cell reports the number of baseline rows removed per dataset.


In [8]:
# Index baseline rows by exact orientation for fast one-to-one matching
base = baseline_all.copy()

# We need stable row ids to remove specific rows
base = base.reset_index(drop=True)
base["_row_id"] = np.arange(len(base), dtype=np.int64)

# Build maps: (dataset, image_l, image_r) -> list of row_ids
def key_lr(ds, l, r):
    return (str(ds).lower(), ensure_jpg(l), ensure_jpg(r))

base_lut = {}
for rid, ds, l, r in zip(base["_row_id"], base["dataset"], base["image_l"], base["image_r"]):
    k = key_lr(ds, l, r)
    base_lut.setdefault(k, []).append(rid)

# Track which baseline row_ids we remove
to_remove = set()

# Iterate eyetracker rows in given order; each ET row removes at most one baseline row
removed_counts = {}  # per dataset

for ds, l, r in zip(eye["dataset"], eye["image_l"], eye["image_r"]):
    ds = str(ds).lower()
    l = ensure_jpg(l)
    r = ensure_jpg(r)

    k_same = (ds, l, r)
    k_swap = (ds, r, l)

    chosen = None

    # 1) Prefer exact orientation match
    lst = base_lut.get(k_same, [])
    while lst and lst[-1] in to_remove:
        lst.pop()
    if lst:
        chosen = lst.pop()

    # 2) If none, allow swapped orientation match
    if chosen is None:
        lst = base_lut.get(k_swap, [])
        while lst and lst[-1] in to_remove:
            lst.pop()
        if lst:
            chosen = lst.pop()

    # Consume at most one baseline row for this ET row
    if chosen is not None:
        to_remove.add(chosen)
        removed_counts[ds] = removed_counts.get(ds, 0) + 1

# Remove only the matched baseline rows (one-to-one)
mask_remove = base["_row_id"].isin(to_remove)
baseline_remaining = base.loc[~mask_remove].drop(columns=["_row_id"]).copy()

# Append eyetracker rows (do NOT swap baseline; eyetracker rows are kept as provided)
combined = pd.concat([baseline_remaining, eye], ignore_index=True)

print("Baseline rows removed (one-to-one with eyetracker rows):")
display(pd.Series(removed_counts).sort_values(ascending=False).to_frame("n_removed"))

print("Combined (baseline + eyetracker):", combined.shape)

Baseline rows removed (one-to-one with eyetracker rows):


,n_removed
berlin,999
sequences,420


Combined (baseline + eyetracker): (13623, 11)


## 8) Multiplicity audit (no row removal)

### Why this step exists

In this project, **each row represents a single human judgment** (one participant response).
Therefore, observing multiple rows with the same:

- `dataset`
- `image_l`
- `image_r`
- `score`
- `has_eyetracker`

**does not automatically indicate a data error**.

In most cases, it means that **multiple independent participants evaluated the same image pair and gave the same answer**.  
This agreement is **meaningful signal**, as it reflects confidence and consistency in perceived safety.

---

### What is *not* done here

This step **does not remove any rows**.

Collapsing identical rows would implicitly convert the dataset from:

> a multiset of human judgments  

into

> a set of unique comparison outcomes

which would **destroy information about agreement strength** and bias the empirical distribution.

---

### Correct deduplication policy

The **only deduplication applied in this pipeline** is performed earlier at the **participant level**, using the key:



In [9]:
audit_subset = ["dataset", "image_l", "image_r", "score", "has_eyetracker"]

# Count how many times each identical training-relevant row appears
row_mult = (
    combined
    .groupby(audit_subset)
    .size()
    .rename("n_rows")
    .reset_index()
)

n_total = len(combined)
n_unique_rows = len(row_mult)
n_rows_with_multiplicity = int((row_mult["n_rows"] > 1).sum())

print("Multiplicity audit (no rows dropped)")
print("--------------------------------------------------------")
print(f"Total rows: {n_total}")
print(f"Unique rows by {audit_subset}: {n_unique_rows}")
print(f"Rows (unique keys) that appear more than once: {n_rows_with_multiplicity}")

# How many *extra* rows are due to multiplicity (i.e., sum(n_rows-1))
extra_mass = int((row_mult["n_rows"] - 1).clip(lower=0).sum())
print(f"Total extra rows beyond unique keys (sum(n_rows-1)): {extra_mass}")

print("\nTop-5 most repeated identical rows (often reflects agreement):")
display(
    row_mult.sort_values("n_rows", ascending=False).head(5)
)

# Per-dataset summary of multiplicity:
# - extra rows beyond one-per-key
# - fraction of dataset rows that are part of repeated keys
per_ds = row_mult.groupby("dataset").agg(
    unique_keys=("n_rows", "size"),
    total_rows=("n_rows", "sum"),
    extra_rows=("n_rows", lambda s: int((s - 1).clip(lower=0).sum())),
    repeated_keys=("n_rows", lambda s: int((s > 1).sum())),
).sort_values("extra_rows", ascending=False)

per_ds["pct_rows_in_repeated_keys"] = (
    (row_mult[row_mult["n_rows"] > 1].groupby("dataset")["n_rows"].sum())
    / per_ds["total_rows"]
).fillna(0.0) * 100.0

print("\nPer-dataset multiplicity summary (no dropping):")
display(per_ds)

# OPTIONAL: If you strongly suspect some are true artifacts (e.g., same source appended twice),
# you can inspect them by filtering where n_rows is very large, or cross-check earlier steps.
combined_final = combined.reset_index(drop=True)
print("\nCombined final shape (unchanged):", combined_final.shape)


Multiplicity audit (no rows dropped)
--------------------------------------------------------
Total rows: 13623
Unique rows by ['dataset', 'image_l', 'image_r', 'score', 'has_eyetracker']: 12430
Rows (unique keys) that appear more than once: 587
Total extra rows beyond unique keys (sum(n_rows-1)): 1193

Top-5 most repeated identical rows (often reflects agreement):


,dataset,image_l,image_r,score,has_eyetracker,n_rows
11030,sequences,fi_3_5.jpg,fi_3_6.jpg,0,False,14
11067,sequences,fi_4_0.jpg,fi_4_1.jpg,0,False,14
11002,sequences,fi_3_0.jpg,fi_3_1.jpg,0,False,14
11104,sequences,fi_4_5.jpg,fi_4_6.jpg,0,False,13
10755,sequences,fi_1_5.jpg,fi_1_6.jpg,0,False,12



Per-dataset multiplicity summary (no dropping):


,unique_keys,total_rows,extra_rows,repeated_keys,pct_rows_in_repeated_keys
dataset,,,,,
sequences,1823,3000,1177,571,58.266667
berlin,7255,7270,15,15,0.412655
london_uk_collideoscope,558,559,1,1,0.357782
barcelona,1153,1153,0,0,0.000000
london_uk_gov,559,559,0,0,0.000000
munich,533,533,0,0,0.000000
paris,549,549,0,0,0.000000



Combined final shape (unchanged): (13623, 11)


## 9) Export training-compatible `comparisons_df.pickle`
We keep only the columns expected by training. Participant/time columns are dropped.

Base columns:
- `dataset`, `image_l`, `image_r`, `score`, `has_eyetracker`

Optional eyetracker metadata (kept if present):
- `survey_id`, `trial_id`, `npy_file_l`, `npy_file_r`


In [10]:
keep_cols = ["dataset","image_l","image_r","score","has_eyetracker"]
for c in ["survey_id","trial_id","npy_file_l","npy_file_r"]:
    if c in combined_final.columns:
        keep_cols.append(c)

train_df = combined_final[keep_cols].copy()

# Normalize again for safety
train_df["dataset"] = train_df["dataset"].astype(str).str.lower()
train_df["image_l"] = train_df["image_l"].map(ensure_jpg)
train_df["image_r"] = train_df["image_r"].map(ensure_jpg)
train_df["score"] = train_df["score"].map(coerce_score)

assert train_df["score"].isna().sum() == 0, "NaN scores found after normalization."

train_df.to_pickle(OUT_PKL)
print(f"Saved training pickle: {OUT_PKL.resolve()}")
print("Saved shape:", train_df.shape)
display(train_df.head(10))


Saved training pickle: /home/csantiago/build_datasets/comparisons_df.pickle
Saved shape: (13623, 9)


,dataset,image_l,image_r,score,has_eyetracker,survey_id,trial_id,npy_file_l,npy_file_r
0,barcelona,381.jpg,1390.jpg,1,False,NaN,NaN,NaN,NaN
1,barcelona,418.jpg,1511.jpg,1,False,NaN,NaN,NaN,NaN
2,barcelona,544.jpg,3226.jpg,1,False,NaN,NaN,NaN,NaN
3,barcelona,374.jpg,5141.jpg,1,False,NaN,NaN,NaN,NaN
4,barcelona,435.jpg,5055.jpg,-1,False,NaN,NaN,NaN,NaN
5,barcelona,613.jpg,4991.jpg,-1,False,NaN,NaN,NaN,NaN
6,barcelona,643.jpg,5250.jpg,-1,False,NaN,NaN,NaN,NaN
7,barcelona,562.jpg,6309.jpg,1,False,NaN,NaN,NaN,NaN
8,barcelona,645.jpg,5730.jpg,-1,False,NaN,NaN,NaN,NaN
9,barcelona,655.jpg,2845.jpg,-1,False,NaN,NaN,NaN,NaN


## 10) Final statistics
This final cell provides statistics you typically need before training:
- rows per dataset
- score distribution overall and per dataset
- eyetracker coverage per dataset
- unique comparisons (ordered) and unique pairs (swap-safe)
- pair multiplicity and image frequency
- tie rates


In [11]:
df = train_df.copy()

print("========================================================")
print("FINAL COMPARISONS DATASET STATISTICS")
print("========================================================")
print("Total rows:", len(df))
print("Number of datasets:", df["dataset"].nunique())

print("\n--- Rows per dataset ---")
display(df["dataset"].value_counts().to_frame("n_rows"))

print("\n--- Overall score distribution ---")
display(df["score"].value_counts().sort_index().to_frame("n_rows"))

print("\n--- Score distribution by dataset (counts) ---")
display(df.groupby(["dataset","score"]).size().unstack(fill_value=0).sort_index())

print("\n--- Eyetracker availability by dataset (%) ---")
display((df.groupby("dataset")["has_eyetracker"].mean() * 100.0).sort_values(ascending=False).to_frame("pct_has_eyetracker"))

print("\n--- Unique comparisons (ordered) ---")
ordered_unique = df.drop_duplicates(subset=["dataset","image_l","image_r","score","has_eyetracker"])
print("Unique ordered rows:", len(ordered_unique))

print("\n--- Unique pairs (swap-safe, ignore score/has_eye) ---")
pair_keys = df.apply(lambda r: unordered_pair_key(r["dataset"], r["image_l"], r["image_r"]), axis=1)
print("Unique swap-safe pairs:", pair_keys.nunique())

print("\n--- Pair multiplicity stats (rows per swap-safe pair) ---")
pair_counts = pair_keys.value_counts()
display(pair_counts.describe().to_frame("pair_count_stats").T)

print("\nTop-20 most repeated swap-safe pairs:")
display(pair_counts.head(20).to_frame("n_rows_for_pair"))

print("\n--- Per-image frequency (left + right) ---")
img_counts = pd.concat(
    [
        df[["dataset","image_l"]].rename(columns={"image_l":"image"}),
        df[["dataset","image_r"]].rename(columns={"image_r":"image"}),
    ],
    ignore_index=True,
)
img_freq = img_counts.groupby(["dataset","image"]).size().sort_values(ascending=False)
print("Top-20 most frequent images:")
display(img_freq.head(20).to_frame("n_appearances"))

print("\n--- Tie rate overall and by dataset ---")
tie_overall = (df["score"] == 0).mean() * 100.0
print(f"Tie rate overall: {tie_overall:.2f}%")
tie_by_ds = (df.assign(is_tie=(df["score"]==0))
               .groupby("dataset")["is_tie"].mean()
               .mul(100.0)
               .sort_values(ascending=False))
display(tie_by_ds.to_frame("pct_ties"))


FINAL COMPARISONS DATASET STATISTICS
Total rows: 13623
Number of datasets: 7

--- Rows per dataset ---


,n_rows
dataset,
berlin,7270
sequences,3000
barcelona,1153
london_uk_gov,559
london_uk_collideoscope,559
paris,549
munich,533



--- Overall score distribution ---


,n_rows
score,
-1,4683
0,3825
1,5115



--- Score distribution by dataset (counts) ---


...

,n_rows_for_pair
"(sequences, fi_4_0.jpg, fi_4_1.jpg)",18
"(sequences, fi_1_5.jpg, fi_1_6.jpg)",18
"(sequences, fi_0_0.jpg, fi_0_1.jpg)",17
"(sequences, fi_4_5.jpg, fi_4_6.jpg)",17
"(sequences, fi_0_5.jpg, fi_0_6.jpg)",17
"(sequences, fi_1_0.jpg, fi_1_1.jpg)",17
"(sequences, fi_2_2.jpg, fi_2_3.jpg)",16
"(sequences, fi_2_0.jpg, fi_2_1.jpg)",15
"(sequences, fi_3_5.jpg, fi_3_6.jpg)",15
"(sequences, fi_3_0.jpg, fi_3_1.jpg)",15



--- Per-image frequency (left + right) ---
Top-20 most frequent images:


n_appearances
dataset   image                    
sequences fi_1_3.jpg             39
          fi_0_2.jpg             38
          fi_4_4.jpg             38
          fi_2_5.jpg             38
          fi_1_5.jpg             38
          fi_4_0.jpg             38
          fi_2_4.jpg             37
          fi_3_0.jpg             37
          fi_2_3.jpg             37
          fi_2_2.jpg             37
          fi_1_4.jpg             37
          fi_0_5.jpg             37
          fi_1_2.jpg             37
          fi_4_3.jpg             37
          fi_4_1.jpg             36
          fi_4_2.jpg             36
          fi_3_2.jpg             36
          fi_0_6.jpg             35
          fi_3_3.jpg             35
          fi_1_6.jpg             35


--- Tie rate overall and by dataset ---
Tie rate overall: 28.08%


,pct_ties
dataset,
sequences,49.566667
london_uk_gov,32.915921
paris,32.604736
london_uk_collideoscope,30.590340
barcelona,28.967910
munich,20.075047
berlin,18.748281
